# 🎬 Supernan AI Intern – Hindi Dubbing Pipeline
**Google Colab Notebook (Free T4 GPU)**

> **Runtime**: Set Runtime → **T4 GPU** before running.
> **Source language**: Kannada (auto-detected). Clip window: **0:30 – 1:00** (30s, speech-dense).

## Cell 1 – Install System Packages

In [ ]:
!apt-get install -y -q ffmpeg rubberband-cli
!ffmpeg -version | head -1
print('✓ System packages ready')

## Cell 2 – Install Python Dependencies

In [ ]:
!pip install -q ffmpeg-python pydub colorlog soundfile pyrubberband numpy
print('✓ Core utilities')

!pip install -q openai-whisper
print('✓ Whisper')

!pip install -q deep-translator
print('✓ deep-translator')

# Coqui XTTS v2 — Python 3.12 compatible community fork
!pip install -q git+https://github.com/idiap/coqui-ai-tts
print('✓ Coqui XTTS v2')

!pip install -q facexlib basicsr gfpgan
print('✓ GFPGAN')

!pip install -q huggingface_hub transformers sentencepiece sacremoses
print('✓ HuggingFace stack')

!pip install -q opencv-python-headless
print('✓ OpenCV')

## Cell 3 – Install IndicTransToolkit

In [ ]:
!pip install -q indictranstoolkit
print('✓ IndicTransToolkit installed')

## Cell 4 – Clone VideoReTalking (Optional — only needed for lip-sync)

In [ ]:
import os
# Since we use --skip-lipsync, Cell 4 can be skipped.
# Run only if you want full lip-sync (Option B in Cell 7).
if not os.path.exists('video-retalking'):
    !git clone https://github.com/vinthony/video-retalking.git

# Checkpoints from public GitHub Release (no HuggingFace auth needed)
if not os.path.exists('video-retalking/checkpoints'):
    print('Downloading VideoReTalking checkpoints (~1 GB) ...')
    !wget -q --show-progress \
        https://github.com/vinthony/video-retalking/releases/download/v0.0.1/checkpoints.zip \
        -O /tmp/vr_checkpoints.zip
    !unzip -q /tmp/vr_checkpoints.zip -d video-retalking/
    !rm /tmp/vr_checkpoints.zip
    print('✓ Checkpoints downloaded')
else:
    print('✓ Checkpoints already present')

!pip install -q -r video-retalking/requirements.txt 2>&1 | tail -2 || true
print('✓ VideoReTalking ready')

## Cell 5 – Clone / Update Pipeline Repo

In [ ]:
import os
REPO_URL = 'https://github.com/sudip-kumar-prasad/supernan-hindi-dubbing-pipeline.git'

if os.path.exists('supernan-hindi-dubbing-pipeline'):
    !git -C supernan-hindi-dubbing-pipeline pull
else:
    !git clone {REPO_URL}

import sys
sys.path.insert(0, '/content/supernan-hindi-dubbing-pipeline')
print('✓ Pipeline repo ready')

## Cell 6 – Download Source Video

In [ ]:
!pip install -q gdown

if not os.path.exists('/content/input.mp4'):
    FILE_ID = '1uDzLVEow_gAJsXnNjbSoskzVLZ4d7opW'
    !gdown {FILE_ID} -O /content/input.mp4
else:
    print('Video already downloaded')

assert os.path.exists('/content/input.mp4'), 'input.mp4 not found'
print(f'✓ Video ready: {os.path.getsize("/content/input.mp4") / 1e6:.1f} MB')

## Cell 7 – Run the Pipeline

> Using **0:30 – 1:00** (30-second window) to capture enough Kannada speech for a 15-20s Hindi dubbed output.

In [ ]:
os.chdir('/content/supernan-hindi-dubbing-pipeline')

# Clear old cache to force fresh run with latest fixes
!rm -rf tmp/

# 30-second clip (0:30–1:00) gives 15-20s of dubbed Hindi output
!python dub_video.py \
    --input /content/input.mp4 \
    --output /content/output.mp4 \
    --start 30 \
    --end 60 \
    --model large-v3 \
    --skip-lipsync \
    --verbose

## Cell 8 – Preview & Download Output

In [ ]:
from IPython.display import Video, display
import os

output_path = '/content/output.mp4'
assert os.path.exists(output_path), 'output.mp4 not found – check Cell 7 logs'
print(f'Output: {os.path.getsize(output_path) / 1e6:.1f} MB')
display(Video(output_path, embed=True, width=640))

In [ ]:
from google.colab import files
files.download('/content/output.mp4')

---
## 💡 Tips
- **30s window (0:30-1:00)** gives ~15-20s of Hindi dubbed output (speech density ~60%)
- If output is still short, try `--start 20 --end 55` or `--start 60 --end 90`
- Source is **Kannada** — `large-v3` Whisper handles it well
- Cell 4 (VideoReTalking) can be **skipped** since we use `--skip-lipsync`